# Data guide

This one notebook explains the data: which files make it, where it comes from, how it is matched, and what X and Y look like.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start this notebook inside the repository.")
    ROOT = ROOT.parent
DATA = Path(os.environ.get("CROPYIELD_DATA_DIR") or ROOT / "data")

## Step 1: Clean structure

The repository holds only the data path. Each file that stays has a name that says its purpose. Everything else is safe in `_parked/`.

**Every file that stays, in pipeline order.** A folder row stands for the files in that folder.

| File | Purpose |
|---|---|
| `README.md` | one screen: what the project is, and the way to this guide |
| `notebooks/data_guide.ipynb` | this guide |
| `pyproject.toml` | the `cropyield` package, its dependencies and the test settings |
| `.gitignore` | keeps chips, the database and other local files out of Git |
| `.gitattributes` | keeps data files and parked files byte-identical in every checkout |
| `scripts/README.md` | every script and when to run it |
| `scripts/get_all_data.py` | one command: runs the data steps in order |
| `scripts/step1_download_labels.py` | provider label files into `data/raw/` |
| `scripts/step1b_download_zimbabwe2024_labels.py` | Zimbabwe 2024 label files |
| `data/raw/` | the provider files, unchanged |
| `data/metadata/` | provider catalogue responses, source URLs and checksums |
| `data/README.txt` | what the data folder holds, with sources and licences |
| `scripts/step2_audit_labels.py` | audit tables in `data/audit/` |
| `data/audit/` | audited label rows, screened candidates and field polygons |
| `src/cropyield/__init__.py` | package root: finds the repository folder |
| `src/cropyield/data/__init__.py` | marks the data subpackage |
| `src/cropyield/data/cohorts.py` | the cohorts, their season windows and their rows from the audit tables |
| `src/cropyield/data/labels.py` | the rules of `labels_v1` and its checks |
| `scripts/step3_build_labels.py` | `data/processed/labels_v1.parquet` |
| `data/processed/labels_v1.parquet` | the one modelling table, frozen |
| `src/cropyield/eval/__init__.py` | marks the evaluation subpackage |
| `src/cropyield/eval/splits.py` | the split rules and their checks |
| `scripts/step4_build_splits.py` | `data/splits/` |
| `data/splits/` | the frozen split files, one per cohort and family |
| `data/splits/manifest.json` | the seed, the row counts and the SHA-256 of `labels_v1` and of each split file |
| `data/splits/README.md` | what a split file holds |
| `configs/sentinel2_windows_pilot48.csv` | the records and season windows of the pilot download |
| `configs/sentinel2_scenes_v1.parquet` | the frozen Sentinel-2 scene of each record and month |
| `scripts/step5_download_imagery.py` | Sentinel-2 chips into `data/imagery/` |
| `src/cropyield/longpath.py` | paths past the Windows length limit |
| `src/cropyield/satellite/__init__.py` | marks the satellite subpackage |
| `src/cropyield/satellite/chips.py` | reads one chip folder |
| `src/cropyield/satellite/records.py` | matches chip folders to labelled records |
| `src/cropyield/satellite/monthly.py` | the monthly band values of one field |
| `src/cropyield/features/__init__.py` | marks the features subpackage |
| `src/cropyield/features/indices.py` | the vegetation indices |
| `sql/schema.sql` | one `CREATE TABLE` per table, each column with a comment |
| `sql/views.sql` | the views `season_indices`, `master` and `monthly` |
| `sql/README.md` | the database, table by table |
| `sql/queries/coverage_by_cohort.sql` | example query: coverage per cohort |
| `sql/queries/modelling_table.sql` | example query: one cohort, one row per field |
| `sql/queries/monthly_series.sql` | example query: one field, month by month |
| `sql/queries/table_overview.sql` | example query: the row count of each table |
| `scripts/step6_build_database.py` | `data/cropyield.duckdb` |
| `src/cropyield/data/database.py` | opens the database from Python |
| `scripts/step7_check_data.py` | what the database holds, per cohort |
| `tests/conftest.py` | test helpers: a small synthetic chip |
| `tests/test_labels.py` | tests of `labels_v1` |
| `tests/test_cohorts.py` | tests of the cohort rows and windows |
| `tests/test_splits.py` | tests of the frozen splits |
| `tests/test_get_data.py` | tests of `scripts/get_all_data.py` |
| `tests/test_download_selection.py` | tests of `scripts/step5_download_imagery.py` |
| `tests/test_longpath.py` | tests of the long paths |
| `tests/test_records.py` | tests of the match between chips and labels |
| `tests/test_monthly.py` | tests of the chips and the monthly values |
| `tests/test_build_database.py` | tests of `scripts/step6_build_database.py` |
| `tests/test_check_data.py` | tests of `scripts/step7_check_data.py` |
| `docs/DATA.md` | how each cohort measures yield, and the `labels_v1` table |
| `docs/EVALUATION.md` | split families and metrics |
| `docs/ROADMAP.md` | the plan |
| `docs/RELATED_WORK.md` | the papers behind the plan |
| `docs/DECISIONS.md` | one dated line per decision |
| `docs/STRUCTURE.md` | folder layout and team rules |

**What went to the parking folder.** `_parked/MANIFEST.csv` lists each file with its SHA-256.

| Parked file | Why it left |
|---|---|
| `_parked/notebooks/00_data_tour.ipynb` | Exploration notebook, not on the data path. notebooks/data_guide.ipynb replaces it. |
| `_parked/notebooks/01_presto_first_run.ipynb` | Exploration notebook, not on the data path. notebooks/data_guide.ipynb replaces it. |
| `_parked/notebooks/02_pilot48.ipynb` | Exploration notebook, not on the data path. notebooks/data_guide.ipynb replaces it. |
| `_parked/notebooks/03_how_the_data_works.ipynb` | Exploration notebook, not on the data path. notebooks/data_guide.ipynb replaces it. |
| `_parked/notebooks/04_database.ipynb` | Exploration notebook, not on the data path. notebooks/data_guide.ipynb replaces it. |
| `_parked/results/README.md` | Model result, not on the data path. |
| `_parked/results/metrics.csv` | Model result, not on the data path. |
| `_parked/configs/README.md` | Experiment or pilot config, not on the data path. |
| `_parked/configs/b0_seasonal_indices_ridge.yaml` | Experiment or pilot config, not on the data path. |
| `_parked/configs/x1_cross_cohort_transfer.yaml` | Experiment or pilot config, not on the data path. |
| `_parked/configs/sentinel2_windows_malawi_2023_pilot.csv` | Experiment or pilot config, not on the data path. |
| `_parked/docs/DATA_Description.md` | Superseded or parked document, not on the data path. |
| `_parked/docs/DATA_SOURCES.md` | Superseded or parked document, not on the data path. |
| `_parked/scripts/run_experiment.py` | Model run, not on the data path. |
| `_parked/scripts/run_transfer.py` | Model run, not on the data path. |
| `_parked/scripts/features/mosaiks_features.py` | MOSAIKS feature step, a model input outside this freeze. |
| `_parked/scripts/imagery/select_pilot.py` | Pilot selection. The rule now lives in scripts/get_all_data.py. |
| `_parked/scripts/models/get_presto.py` | Presto weights step, outside this freeze. |
| `_parked/src/cropyield/eval/metrics.py` | Model scores, not on the data path. |
| `_parked/src/cropyield/features/mosaiks.py` | MOSAIKS feature code, a model input outside this freeze. |
| `_parked/src/cropyield/features/presto_inputs.py` | Presto input code, outside this freeze. |
| `_parked/tests/test_metrics.py` | Test of a parked file. |
| `_parked/tests/test_monthly_presto_inputs.py` | Parked as it was. Its monthly tests are cut into tests/test_monthly.py. |
| `_parked/tests/test_mosaiks.py` | Test of a parked file. |
| `_parked/tests/test_mosaiks_features.py` | Test of a parked file. |
| `_parked/tests/test_presto_model.py` | Test of a parked file. |
| `_parked/tests/test_run_experiment.py` | Test of a parked file. |
| `_parked/tests/test_run_transfer.py` | Test of a parked file. |
| `_parked/docs/archive/` | Superseded or parked document, not on the data path. |
| `_parked/scripts/cluster/` | Cluster launcher of old script names, not on the data path. |
| `_parked/src/cropyield/models/` | Model code, outside this freeze. |

**To take a file back:** the old path is the parked path without its first folder. Run `git mv _parked/<old_path> <old_path>`, and delete the row of the file in `_parked/MANIFEST.csv`. `_parked/LANDING.md` has the details.

In [ ]:
import csv
import hashlib
import json
import re

guide = json.loads((ROOT / "notebooks" / "data_guide.ipynb").read_text(encoding="utf-8"))
step1, inside = [], False
for cell in guide["cells"]:
    text = "".join(cell["source"])
    if cell["cell_type"] == "markdown" and text.startswith("## "):
        inside = text.startswith("## Step 1")
    if inside and cell["cell_type"] == "markdown":
        step1.append(text)
listed = re.findall(r"^\| `([^`]+)` \|", "\n".join(step1), flags=re.MULTILINE)
missing = [path for path in listed if not (ROOT / path).exists()]

with open(ROOT / "_parked" / "MANIFEST.csv", encoding="utf-8", newline="") as handle:
    parked = list(csv.DictReader(handle))
changed = [row["new_path"] for row in parked
           if hashlib.sha256((ROOT / row["new_path"]).read_bytes()).hexdigest() != row["sha256"]]

assert not missing, f"listed but missing: {missing}"
assert not changed, f"parked file changed: {changed}"
print(f"Step 1 holds: all {len(listed)} listed paths exist, and all {len(parked)} parked files keep their hash.")

## Step 2: Sources and download

Each dataset comes from a public source. The table below names the provider, the link, the licence and the script for each one. The cell under it states the real cost of a download.

**What is in Git.** All label data: `data/raw/`, `data/metadata/`, `data/audit/`, `data/processed/` and `data/splits/`. A clone holds all of it.

**What you must download.** The Sentinel-2 chips. They are not in Git. A download puts them in `data/imagery/`. The database build then makes `data/cropyield.duckdb` from the labels and the chips. The photo archive of the Zimbabwe 2024 source is not in Git, and the data path does not need it.

**What the same local copy means.** Every team member has the same labels, splits, chips and indices. Git gives the labels and the splits. The download gives the chips, and the database build computes the indices from the chips. MOSAIKS and Presto are not in this copy. They wait for their model packets.

The downloader asks Earth Search for the scenes of each record, so a change in the catalogue can change a chip. The last cell of this step compares your scenes with the frozen list, and names each record that differs.

In [ ]:
import json
import sys
from urllib.parse import urlparse

import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(ROOT / "scripts"))
import get_all_data

STAC = "https://earth-search.aws.element84.com/v1"
ARCHIVE = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/"
COLLECTION = "sentinel-2-c1-l2a"
LABEL_SCRIPT = "scripts/step1_download_labels.py"

# provider, URL, licence, cohort, what the data is, local path, script
LABEL_SOURCES = [
    ("One Acre Fund", "https://data.harvestportal.org/dataset/667320b8-6a39-4e42-8b29-25a2fa91d2fd", "CC BY",
     "kenya_oaf_2019_2020", "monitoring workbook: crop cuts with GPS points", "data/raw/one_acre_fund_full", LABEL_SCRIPT),
    ("One Acre Fund", "https://zenodo.org/records/11122388", "CC BY 4.0", "none: it overlaps the full workbook",
     "curated maize release", "data/raw/one_acre_fund_curated", LABEL_SCRIPT),
    ("Lacuna", "https://zenodo.org/records/10060610", "CC BY 4.0", "zimbabwe_lacuna_2022_2023",
     "crop cuts with GPS points, and the technical manual", "data/raw/zimbabwe", LABEL_SCRIPT),
    ("FAO / NASA Harvest", "https://data.harvestportal.org/dataset/61615c12-2724-4f56-9385-4eb09eabea64", "CC BY",
     "malawi_2022, malawi_2023", "crop cuts with walked field polygons", "data/raw/malawi", LABEL_SCRIPT),
    ("FAO / NASA Harvest", "https://data.harvestportal.org/dataset/8c1d3892-abab-4008-9986-4c21adc24518", "CC BY",
     "malawi_2022, malawi_2023", "field dictionaries", "data/raw/malawi_field_dictionary", LABEL_SCRIPT),
    ("FAO / NASA Harvest", "https://data.harvestportal.org/dataset/53a79b82-bfc4-4ba3-9232-4bedb3ab06fa", "CC BY",
     "malawi_2022, malawi_2023", "dataset metadata", "data/raw/malawi_metadata", LABEL_SCRIPT),
    ("Vision-guided yield dataset", "https://zenodo.org/records/17626117", "CC BY 4.0", "zimbabwe_2024",
     "plot yields, coordinates and crop-cut details", "data/raw/vision_guided_yield_estimation",
     "scripts/step1b_download_zimbabwe2024_labels.py"),
]

metadata = DATA / "metadata"
manifest = json.loads((metadata / "download_manifest.json").read_text(encoding="utf-8"))
vision = json.loads((metadata / "vision_guided_yield_download_manifest.json").read_text(encoding="utf-8"))
label_files = manifest["files"] + [vision["files"]["metadata.csv"]]
manifest_urls = [item["url"] for item in manifest["files"]] + [vision["record_url"]]
folders = [source[5] for source in LABEL_SOURCES]

# The table cannot drift from the manifests: each file has a row, and each URL leads to its files.
assert all(any(item["path"].startswith(folder + "/") for folder in folders) for item in label_files)
assert all(any(source[1].rsplit("/", 1)[-1] in url for url in manifest_urls) for source in LABEL_SOURCES)
assert {urlparse(url).hostname for url in manifest_urls} <= {urlparse(source[1]).hostname for source in LABEL_SOURCES}

downloader = get_all_data.step5_download_imagery
assert (downloader.API, downloader.COLLECTION) == (STAC, COLLECTION)
fields = len(pd.read_parquet(DATA / "processed" / "labels_v1.parquet", columns=["obs_id"]))
bands = ", ".join(downloader.BANDS)

table = [["Provider", "URL", "Licence", "Cohort", "What the data is", "Local path", "Size", "Script"],
         ["---"] * 8]
for provider, url, licence, cohort, what, folder, script in LABEL_SOURCES:
    size = sum(item["bytes"] for item in label_files if item["path"].startswith(folder + "/")) / 1e6
    table.append([provider, url, licence, cohort, what, f"`{folder}/`", f"{size:.2f} MB", f"`{script}`"])
table.append(["Element 84 Earth Search on AWS", f"STAC: {STAC}<br>archive: {ARCHIVE}", "Copernicus Sentinel data terms",
              "all cohorts", f"collection `{COLLECTION}`, bands {bands} and SCL, as chips around each field",
              "`data/imagery/sentinel2/`", f"about {fields * get_all_data.MB_PER_RECORD / 1000:.1f} GB",
              "`scripts/get_all_data.py`, which calls `scripts/step5_download_imagery.py`"])
display(Markdown("\n".join("| " + " | ".join(row) + " |" for row in table)))

In [ ]:
pilot = len(pd.read_csv(ROOT / "configs" / "sentinel2_windows_pilot48.csv"))
print(f"Per field: about {get_all_data.MB_MOVED_PER_RECORD} MB moved over your line, "
      f"about {get_all_data.MB_PER_RECORD} MB kept on disk.")
print(f"Pilot test: {get_all_data.cost_text(pilot)}.")
print(f"All data:   {get_all_data.cost_text(fields)}.")

**Get the data.** Each of the next two cells is a button. Press the Run arrow of one cell. The first cell is the pilot test, and the second cell is all the data. Do the pilot test first.

- **Switch.** `DOWNLOAD` at the top of the cell turns the cell on or off. When you have the data, set it to `False`, so that a rerun skips the cell and asks nothing.
- **Typed yes.** The cell asks you to type `yes`. Any other answer stops it, so Run All cannot start a download by accident.
- **Speed.** `SLICES` at the top of the cell sets how many parallel slices run in the background. More slices help only until your line is full.
- **Progress.** A loading bar shows the percent done, the fields done, the GB on disk and the time left. The logs are in the `logs` folder beside the chips.
- **Stop and restart.** You can stop at any time with the stop button of the cell. Every finished field stays on disk. When you run the cell again, it checks the disk and downloads only the missing fields. A failed field gets a new try.
- **Database.** After the chips, the cell builds the database, so close other notebooks that have it open. If the database holds rows in `presto_embeddings`, the cell first says that the build removes them, and asks you to type `yes`.
- **Result.** The last two lines say whether every record is complete, and how many records hold the same scenes as the frozen list.

In [ ]:
DOWNLOAD = True  # the switch: set it to False to skip this cell on a rerun
SLICES = 32      # parallel slices: lower it on a slow line

# The pilot test. This cell also works alone in a new kernel.
import os
import sys
from pathlib import Path

if DOWNLOAD:
    ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "pyproject.toml").is_file())
    sys.path.insert(0, str(ROOT / "scripts"))
    import get_all_data

    get_all_data.notebook_download("pilot", SLICES, Path(os.environ.get("CROPYIELD_DATA_DIR") or ROOT / "data"))
else:
    print("Download is off. Set DOWNLOAD = True at the top of this cell to turn it on.")

In [ ]:
DOWNLOAD = True  # the switch: set it to False to skip this cell on a rerun
SLICES = 32      # parallel slices: lower it on a slow line

# All data. This cell also works alone in a new kernel.
import os
import sys
from pathlib import Path

if DOWNLOAD:
    ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "pyproject.toml").is_file())
    sys.path.insert(0, str(ROOT / "scripts"))
    import get_all_data

    get_all_data.notebook_download("all", SLICES, Path(os.environ.get("CROPYIELD_DATA_DIR") or ROOT / "data"))
else:
    print("Download is off. Set DOWNLOAD = True at the top of this cell to turn it on.")

**Compare your scenes.** This cell needs no network. It compares the `scenes` table of the database with `configs/sentinel2_scenes_v1.parquet`, record by record. It counts only the records on disk, so the pilot test can pass. A different `item_id` or a missing month makes a record differ. The cell reports a difference and changes nothing.

A clean download can still differ. In the first download, a local file error stopped the first-choice scene of a few records, and the downloader took the next scene of that month. A clean download takes the first choice, so those records appear here.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from cropyield.data.database import scene_differences

SCENE_LIST = ROOT / "configs" / "sentinel2_scenes_v1.parquet"
SHOWN = 20  # the first records that differ; the list `differ` holds them all

database = DATA / "cropyield.duckdb"
if database.exists():
    on_disk, differ = scene_differences(database, SCENE_LIST)
    print(f"{on_disk - len(differ):,} of {on_disk:,} records on disk hold the same scenes as the frozen list.")
    for obs_id in differ[:SHOWN]:
        print(f"Differs: {obs_id}")
    if len(differ) > SHOWN:
        print(f"And {len(differ) - SHOWN:,} more records. The list `differ` holds them all.")
else:
    print("No database yet, so there is nothing to compare. Run a download cell first.")

## Step 3: Harmonization

This step will show how the labels and the Sentinel-2 data are matched.

## Step 4: X and Y

This step will show what X and Y are, and every known flaw in them.